# Day 059 — Exercise 2: JobQueue Class

`BackgroundTasks` is simple but limited: results aren't persistently stored, and you can't check status later. For real job tracking you need a dedicated queue that remembers every job's state.

`threading.Lock` ensures only one thread reads or writes the `_jobs` dict at a time. Without it, two threads updating the same dict simultaneously can corrupt state.

In [ ]:
import threading
import time
import uuid
from typing import Any, Callable


## Task

Implement `JobQueue` with four behaviours:

| Method | Behaviour |
|--------|-----------|
| `submit(fn, *args) -> str` | Launch thread, return `job_id` immediately |
| `status(job_id) -> str` | `'pending'/'running'/'done'/'error'/'not_found'` |
| `result(job_id) -> Any` | Return result when done, else `None` |
| `__len__() -> int` | Total jobs ever submitted |

Use `threading.Lock()` around every `_jobs` read/write.

## Your Implementation

In [ ]:
class JobQueue:
    """Thread-safe in-memory job queue.

    submit(fn, *args) -> str
        Run fn(*args) in a background thread. Return job_id.
    status(job_id) -> str
        One of: 'pending', 'running', 'done', 'error', 'not_found'
    result(job_id) -> Any
        Return the result if status is 'done', else None.
    __len__() -> int
        Total number of jobs ever submitted.
    """

    def __init__(self):
        # TODO: init _jobs dict and threading.Lock
        raise NotImplementedError

    def submit(self, fn: Callable, *args) -> str:
        raise NotImplementedError

    def status(self, job_id: str) -> str:
        raise NotImplementedError

    def result(self, job_id: str) -> Any:
        raise NotImplementedError

    def __len__(self) -> int:
        raise NotImplementedError


In [ ]:
class JobQueue:
    def __init__(self):
        self._jobs: dict = {}
        self._lock = threading.Lock()

    def submit(self, fn: Callable, *args) -> str:
        job_id = uuid.uuid4().hex[:8]
        with self._lock:
            self._jobs[job_id] = {"status": "pending"}

        def worker():
            with self._lock:
                self._jobs[job_id]["status"] = "running"
            try:
                value = fn(*args)
                with self._lock:
                    self._jobs[job_id] = {"status": "done", "result": value}
            except Exception as exc:
                with self._lock:
                    self._jobs[job_id] = {"status": "error", "error": str(exc)}

        threading.Thread(target=worker, daemon=True).start()
        return job_id

    def status(self, job_id: str) -> str:
        with self._lock:
            return self._jobs.get(job_id, {}).get("status", "not_found")

    def result(self, job_id: str) -> Any:
        with self._lock:
            job = self._jobs.get(job_id, {})
            return job.get("result") if job.get("status") == "done" else None

    def __len__(self) -> int:
        with self._lock:
            return len(self._jobs)


## Automated checks

In [ ]:
score, total = 0, 5

def _wait(q, job_id, timeout=2.0):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if q.status(job_id) not in ("pending", "running"):
            return
        time.sleep(0.01)

try:
    q = JobQueue()

    # submit and wait
    jid = q.submit(lambda: 42)
    _wait(q, jid)
    assert q.status(jid) == "done", f"Expected done, got {q.status(jid)}"
    assert q.result(jid) == 42, f"Expected 42, got {q.result(jid)}"
    score += 1; print("\u2705 submit runs fn and stores result")

    # result None while not done
    barrier = threading.Event()
    def _slow_fn():
        barrier.wait()
        return "slow_result"
    jid2 = q.submit(_slow_fn)
    time.sleep(0.02)
    assert q.status(jid2) in ("pending", "running"), f"Got {q.status(jid2)}"
    assert q.result(jid2) is None, f"Got {q.result(jid2)}"
    barrier.set()
    _wait(q, jid2)
    assert q.result(jid2) == "slow_result", f"Got {q.result(jid2)}"
    score += 1; print("\u2705 result() returns None while running")

    # unknown job
    assert q.status("nope") == "not_found", f"Got {q.status('nope')}"
    score += 1; print("\u2705 unknown job_id → 'not_found'")

    # error job
    def boom(): raise ValueError("oops")
    jid3 = q.submit(boom)
    _wait(q, jid3)
    assert q.status(jid3) == "error", f"Got {q.status(jid3)}"
    score += 1; print("\u2705 failing fn → status 'error'")

    # __len__
    assert len(q) >= 3, f"Expected at least 3 jobs, got {len(q)}"
    score += 1; print("\u2705 __len__ counts submitted jobs")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class JobQueue:
    def __init__(self):
        self._jobs: dict = {}
        self._lock = threading.Lock()

    def submit(self, fn: Callable, *args) -> str:
        job_id = uuid.uuid4().hex[:8]
        with self._lock:
            self._jobs[job_id] = {"status": "pending"}

        def worker():
            with self._lock:
                self._jobs[job_id]["status"] = "running"
            try:
                value = fn(*args)
                with self._lock:
                    self._jobs[job_id] = {"status": "done", "result": value}
            except Exception as exc:
                with self._lock:
                    self._jobs[job_id] = {"status": "error", "error": str(exc)}

        threading.Thread(target=worker, daemon=True).start()
        return job_id

    def status(self, job_id: str) -> str:
        with self._lock:
            return self._jobs.get(job_id, {}).get("status", "not_found")

    def result(self, job_id: str) -> Any:
        with self._lock:
            job = self._jobs.get(job_id, {})
            return job.get("result") if job.get("status") == "done" else None

    def __len__(self) -> int:
        with self._lock:
            return len(self._jobs)
```

**Why `threading.Lock()`?** The `_jobs` dict is accessed from both the main thread (via `status`/`result`) and background threads (via `worker`). Without a lock, two threads can interleave dictionary writes and corrupt the state — a race condition that is hard to reproduce but catastrophic in production.

</details>